# V6: five-fold YOLO + ResNet34 ensemble without wall-clock limits

**Prepared for a separate longer run; no automatic launch or submission.**
V5 is unchanged. Verified public PQ: V1 0.29, focused V2 0.32, repaired V3 0.35.
V5's fourth scored-experiment result is pending; V4 has no reported score.

Each of five observation-day-grouped folds trains **25 YOLO11s-seg epochs and
20 ResNet34/U-Net refiner epochs**: 125 detector and 100 refiner epochs total.
There are **no coded training deadlines or fold time shares**. Early stopping
is retained: YOLO validation-fitness patience 15; refiner tuning-loss patience 8,
with minimum significant decrease 1e-4. Best checkpoints are still selected
by the existing tuning criteria.
Kaggle session/GPU limits, errors, or manual cancellation can still interrupt
the run. This is not a two-hour configuration and may exceed a single session.

The architecture, day grouping, annotator-specific targets, inner calibration,
and equal-weight five-model ensemble are unchanged from V5. Approximate
partitions are 72% training / 8% inner tuning / 20% outer evaluation.
Outer labels do not select checkpoints, thresholds, or voting weights.

All five models predict each competition image. One-to-one instance matching
uses IoU >= 0.5; retained instances and pixels require 3 of 5 fold votes.
Final disjoint ownership and CSV validation prevent overlapping masks.
Majority voting and longer training do not guarantee higher public PQ.
Only FINAL CSV PREPARED indicates readiness for manual submission.

## T4x2 crop throughput
YOLO remains on GPU 0. ResNet34 training and batched inference use up to both
visible T4 GPUs through DataParallel (batch 16 total, not per GPU). Up to two
CPU loading workers per loader use pinned memory and persistent processes.
Prepared uint8 crops are cached on disk before random augmentation and shared
across folds; only the assigned training records enter each fold's loader.
The first crop pass still pays decoding/cache-write costs. Crop predictions
are processed in batches of up to 16, preserving detector order.
Device availability is printed explicitly. This is not parallel fold training,
TPU support, or proof of a 3–4-hour runtime. DataParallel splits BatchNorm
statistics by device and can change training behavior relative to one GPU.


In [ ]:
import time
CV_STARTED = time.monotonic()


In [ ]:
%pip install -q 'ultralytics==8.4.211' 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy 'opencv-python-headless>=4.9,<5' PyYAML

In [ ]:
import csv
import hashlib
import importlib.metadata
import itertools
import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from PIL import Image
import pycocotools.mask as mask_utils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(seed=2026, image_size=512, batch_size=8, epochs=20,
           learning_rate=3e-4, encoder='resnet18', architecture='Unet',
           pretrained=True, workers=0, tune_fraction=0.15, holdout_fraction=0.15,
           tune_image_limit=32, thresholds=[0.35, 0.5, 0.65],
           min_areas=[64, 256], contrast_thresholds=[0.025, 0.04, 0.06])
RUN_NAME = 'v6_fivefold_resnet34_no_time_limit_seed2026'
RUN_TRAINING = False
RUN_CALIBRATION = False
SMOKE_TEST = False
SUBMIT_TO_KAGGLE = False
DATA_ROOT_OVERRIDE = None
CHECKPOINT_INPUT_DIR = None  # Path to an attached experiment folder for inference-only reproduction.
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else Path.cwd() / 'cv_outputs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else OUTPUT_DIR
CHECKPOINT_PATH = INPUT_DIR / 'best_model.pt'
COMPETITION = 'filament-segmentation-2026'
DATASET_NAME = 'MAGFiLO_1.0_Kaggle_2026'
ANNOTATION_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
FULL_SHAPE = (2048, 2048)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('WARNING: CPU selected. Enable a Kaggle GPU before full training.')
if CFG['image_size'] % 32 or CFG['batch_size'] < 2 or CFG['epochs'] < 1:
    raise ValueError('Use image_size divisible by 32, batch_size >= 2, and epochs >= 1.')
print('Device:', DEVICE, 'outputs:', OUTPUT_DIR)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))

CV_ROOT = OUTPUT_DIR
FOLDS_TO_RUN = [0, 1, 2, 3, 4]  # 0 through 4; change only this to run another identical-plan fold.
INNER_TUNE_FRACTION = 0.1
COLLECT_ROOTS = []  # Paths to downloaded/attached matching fold directories.



In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')

annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
print('Dataset:', len(train_paths), 'labeled physical images;', len(test_paths), 'unlabeled test images')


In [ ]:
@lru_cache(maxsize=128)
def small_image(filename):
    with Image.open(train_lookup[filename]) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{filename}: expected 2048 x 2048 grayscale JPEG.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0

def ann_rle(annotation):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f'Invalid polygon: {annotation["id"]}')
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation['size']) != list(FULL_SHAPE):
            raise ValueError('RLE dimensions do not match the image.')
        if isinstance(segmentation['counts'], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation['counts']
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode('ascii') if isinstance(counts, str) else counts)
    else:
        raise TypeError(f'Unsupported segmentation: {annotation["id"]}')
    if mask_utils.area(rle) <= 0:
        raise ValueError(f'Empty ground-truth filament: {annotation["id"]}')
    return rle

@lru_cache(maxsize=512)
def gt_rles(image_id):
    return [ann_rle(a) for a in annotations_by_id[image_id]]

def union_mask(rles):
    if not rles:
        return np.zeros(FULL_SHAPE, dtype=np.uint8)
    return mask_utils.decode(mask_utils.merge(rles))

@lru_cache(maxsize=128)
def small_target(image_id):
    full_mask = union_mask(gt_rles(image_id))
    resized = Image.fromarray(full_mask).resize((CFG['image_size'], CFG['image_size']), Image.Resampling.NEAREST)
    return np.asarray(resized, dtype=np.float32)

class FilamentDataset(Dataset):
    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = small_image(record['file_name'])
        target = small_target(record['id'])
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        x = torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None]
        y = torch.from_numpy(np.ascontiguousarray(target))[None]
        return x, y



In [ ]:
def build_model(config, pretrained=False):
    constructors = {'Unet': smp.Unet, 'FPN': smp.FPN}
    if config['architecture'] not in constructors:
        raise ValueError('Supported architectures: Unet, FPN.')
    return constructors[config['architecture']](encoder_name=config['encoder'],
        encoder_weights='imagenet' if pretrained else None, in_channels=1, classes=1, activation=None)

def segmentation_loss(logits, target):
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice



In [ ]:
@torch.inference_mode()
def predict_probability(path):
    with Image.open(path) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{path.name}: invalid image mode or dimensions.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np.ascontiguousarray((array - 0.5) / 0.5))[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f'Nonfinite prediction: {path.name}')
    return probability

def full_probability(probability):
    resized = Image.fromarray(np.asarray(probability, dtype=np.float32)).resize(
        (2048, 2048), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def contrast_foreground(filename, threshold):
    image = small_image(filename)
    disk = ndi.binary_erosion(image > 0.08, iterations=max(1, CFG['image_size']//128))
    background = ndi.gaussian_filter(image, sigma=CFG['image_size']/32)
    return disk & ((background - image) > threshold)

def instance_rles(probability, threshold, min_area):
    if min_area < 1 or not 0 <= threshold <= 1:
        raise ValueError('Invalid threshold or minimum area.')
    binary = full_probability(probability) >= threshold
    labels, count = ndi.label(binary, structure=np.ones((3, 3), dtype=np.uint8))
    areas = np.bincount(labels.ravel())
    result = []
    for label in range(1, count+1):
        if areas[label] >= min_area:
            mask = np.asfortranarray(labels == label, dtype=np.uint8)
            result.append(mask_utils.encode(mask))
    return result

def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0]*n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        # Maximum cardinality first, then overlap; greedy matching can lose true positives.
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched); fp = n_pred-tp; fn = n_true-tp
    denominator = tp + 0.5*fp + 0.5*fn
    pq = sum(matched)/denominator if denominator else 1.0
    pred_union, true_union = union_mask(predicted).astype(bool), union_mask(truth).astype(bool)
    intersection = np.count_nonzero(pred_union & true_union)
    union = np.count_nonzero(pred_union | true_union)
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2*intersection/pixels if pixels else 1.0,
                iou=intersection/union if union else 1.0, tp=tp, fp=fp, fn=fn,
                split_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=0) > 1)),
                merge_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=1) > 1)))

def evaluate(names, parameters, probability_cache=None):
    rows = []
    for filename in names:
        if parameters['method'] == 'unet':
            probability = (probability_cache[filename] if probability_cache is not None
                           else predict_probability(train_lookup[filename]))
            threshold = parameters['threshold']
        elif parameters['method'] == 'contrast':
            probability = contrast_foreground(filename, parameters['contrast_threshold']).astype(np.float32)
            threshold = 0.5
        else:
            raise ValueError('Unknown segmentation method.')
        predicted = instance_rles(probability, threshold, parameters['min_area'])
        for record in records_by_name[filename]:
            rows.append(dict(filename=filename, image_id=record['id'],
                             **local_scores(predicted, gt_rles(record['id']))))
    if not rows:
        raise ValueError('Evaluation set is empty.')
    # Average annotators per image, then physical images; do not overweight triply labeled images.
    grouped = defaultdict(list)
    for row in rows: grouped[row['filename']].append(row)
    metrics = {key: float(np.mean([np.mean([r[key] for r in group]) for group in grouped.values()]))
               for key in ['pq', 'dice', 'iou', 'tp', 'fp', 'fn', 'split_relations', 'merge_relations']}
    return metrics, rows


In [ ]:
"""Disjoint instance masks and an offline repair command for saved V3 CSVs."""

import argparse
import csv
import hashlib
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils


MASK_POLICY = "first-mask-wins-v1"
SUBMISSION_FIELDS = ["filament_id", "segmentation_rle"]


def decode_instance(rle, shape):
    if tuple(rle["size"]) != tuple(shape):
        raise ValueError(f"Instance shape {rle['size']} does not match {shape}.")
    mask = mask_utils.decode(rle)
    if mask.shape != tuple(shape) or not np.isin(mask, [0, 1]).all() or not mask.any():
        raise ValueError("An instance must be a nonempty binary mask.")
    return mask.astype(bool)


def disjoint_instances(instances, shape=(2048, 2048), min_area=1):
    """Keep earlier masks unchanged; later masks may claim only unowned pixels."""
    if min_area < 1:
        raise ValueError("min_area must be positive.")
    occupied = np.zeros(shape, dtype=bool)
    result, kept_indices, dropped_indices = [], [], []
    overlap_pixels = 0
    for index, rle in enumerate(instances):
        mask = decode_instance(rle, shape)
        overlap_pixels += int(np.count_nonzero(mask & occupied))
        mask &= ~occupied
        if np.count_nonzero(mask) < min_area:
            dropped_indices.append(index)
            continue
        occupied |= mask
        result.append(mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))))
        kept_indices.append(index)
    return result, dict(
        policy=MASK_POLICY, overlap_pixels_removed=overlap_pixels,
        kept_indices=kept_indices, dropped_indices=dropped_indices,
    )


def validate_disjoint(instances, shape=(2048, 2048)):
    occupied = np.zeros(shape, dtype=bool)
    for index, rle in enumerate(instances):
        mask = decode_instance(rle, shape)
        if np.any(occupied & mask):
            raise ValueError(f"Instance {index} overlaps an earlier instance.")
        occupied |= mask


def submission_groups(rows, test_stems, shape):
    groups, seen = defaultdict(list), set()
    for row in rows:
        if set(row) != set(SUBMISSION_FIELDS):
            raise ValueError("CSV must have exactly the two competition columns.")
        identity = row["filament_id"]
        stem, separator, number = identity.rpartition("_")
        if (not separator or not number.isdigit() or int(number) < 1
                or stem not in test_stems or identity in seen):
            raise ValueError(f"Unknown, invalid, or duplicate filament_id: {identity}")
        seen.add(identity)
        rle = dict(size=list(shape), counts=row["segmentation_rle"].encode("ascii"))
        groups[stem].append((row, rle))
    if not groups:
        raise ValueError("Submission has no predicted instances.")
    return groups


def validate_submission_rows(rows, test_stems, shape=(2048, 2048)):
    groups = submission_groups(rows, test_stems, shape)
    for stem, entries in groups.items():
        try:
            validate_disjoint([rle for _, rle in entries], shape)
        except ValueError as error:
            raise ValueError(f"{stem}: {error}") from error
    return dict(
        rows=len(rows), images_with_predictions=len(groups),
        zero_detection_images=sorted(set(test_stems) - set(groups)),
        overlapping_pixels=0,
    )


def repair_submission(source, destination, test_stems, min_area=20, shape=(2048, 2048)):
    source, destination = Path(source), Path(destination)
    audit_path = destination.with_suffix(".audit.json")
    if source.resolve() == destination.resolve():
        raise ValueError("Repair must not overwrite the original submission.")
    if destination.exists() or audit_path.exists():
        raise FileExistsError("Choose a new repair output path; existing evidence is preserved.")
    with source.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != SUBMISSION_FIELDS:
            raise ValueError("Unexpected submission columns or column order.")
        original_rows = list(reader)
    groups = submission_groups(original_rows, test_stems, shape)
    repaired_by_id, image_audits = {}, []
    for stem, entries in groups.items():
        masks, audit = disjoint_instances([rle for _, rle in entries], shape, min_area)
        for index, rle in zip(audit["kept_indices"], masks):
            identity = entries[index][0]["filament_id"]
            repaired_by_id[identity] = dict(
                filament_id=identity, segmentation_rle=rle["counts"].decode("ascii"),
            )
        image_audits.append(dict(
            image=stem, overlap_pixels_removed=audit["overlap_pixels_removed"],
            input_instances=len(entries), output_instances=len(masks),
            dropped_ids=[entries[i][0]["filament_id"] for i in audit["dropped_indices"]],
        ))
    rows = [repaired_by_id[r["filament_id"]] for r in original_rows
            if r["filament_id"] in repaired_by_id]
    validation = validate_submission_rows(rows, test_stems, shape)
    temporary = destination.with_suffix(".tmp.csv")
    if temporary.exists():
        raise FileExistsError(f"Temporary file already exists: {temporary}")
    try:
        with temporary.open("x", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)
            writer.writeheader()
            writer.writerows(rows)
        with temporary.open(newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != SUBMISSION_FIELDS or list(reader) != rows:
                raise ValueError("Repaired CSV readback mismatch.")
        report = dict(
            policy=MASK_POLICY, min_area=min_area, validation=validation,
            input_csv=str(source), input_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
            output_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),
            input_rows=len(original_rows), dropped_rows=len(original_rows) - len(rows),
            images_with_overlap=sum(a["overlap_pixels_removed"] > 0 for a in image_audits),
            overlap_pixels_removed=sum(a["overlap_pixels_removed"] for a in image_audits),
            images=image_audits,
            note="Offline repair only. No retraining, recalibration, or submission. "
                 "Previous tuning scores do not measure these repaired masks.",
        )
        with audit_path.open("x") as stream:
            stream.write(json.dumps(report, indent=2, allow_nan=False))
        temporary.replace(destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("source", type=Path)
    parser.add_argument("destination", type=Path)
    parser.add_argument("--test-dir", type=Path, required=True)
    parser.add_argument("--min-area", type=int, default=20)
    args = parser.parse_args()
    test_stems = {p.stem for p in args.test_dir.rglob("*.jpeg")}
    if not test_stems:
        parser.error("--test-dir contains no test JPEGs.")
    report = repair_submission(args.source, args.destination, test_stems, args.min_area)
    print(json.dumps({key: value for key, value in report.items() if key != "images"}, indent=2))
    print("REPAIRED CSV:", args.destination, "No competition upload performed.")



In [ ]:
import os
import importlib

runtime_dir = CV_ROOT / "_runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)
(runtime_dir / "v6_crop_runtime.py").write_text('"""CPU crop caching and batched refiner inference for the standalone V6 notebook."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\n\ndef crop_bounds(mask, context=1.8, minimum=96):\n    ys, xs = np.where(mask)\n    if not len(xs):\n        raise ValueError("Cannot crop an empty instance.")\n    height, width = mask.shape\n    side = min(height, width, max(minimum, int(np.ceil(\n        max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))))\n    x = int(np.clip(round((xs.max() + xs.min() + 1 - side) / 2), 0, width - side))\n    y = int(np.clip(round((ys.max() + ys.min() + 1 - side) / 2), 0, height - side))\n    return x, y, x + side, y + side\n\n\ndef normalized_crop(image, bounds, size):\n    array = np.asarray(image.crop(bounds).resize(\n        (size, size), Image.Resampling.BILINEAR), dtype=np.float32) / 255\n    return np.repeat(((array - 0.5) / 0.5)[None], 3, axis=0).copy()\n\n\nclass CachedInstanceCrops(Dataset):\n    """Cache compact pre-augmentation uint8 image/mask pairs, shared across folds."""\n\n    def __init__(self, entries, cache_dir, size=256, context=1.8, minimum=96, augment=False):\n        self.size, self.context, self.minimum = size, context, minimum\n        self.augment = augment\n        directory = Path(cache_dir)\n        directory.mkdir(parents=True, exist_ok=True)\n        self.entries = []\n        for path, rle in entries:\n            path = Path(path)\n            stat = path.stat()\n            counts = rle["counts"]\n            if not isinstance(counts, bytes):\n                raise TypeError("Cached crop entries require compressed byte-count RLEs.")\n            identity = json.dumps([\n                str(path.resolve()), stat.st_size, stat.st_mtime_ns, list(rle["size"]),\n                size, context, minimum, "uint8-bilinear-nearest-v1",\n            ]).encode() + counts\n            key = hashlib.sha256(identity).hexdigest()\n            self.entries.append((path, rle, directory / f"{key}.npy"))\n\n    def __len__(self):\n        return len(self.entries)\n\n    def __getitem__(self, index):\n        path, rle, cached = self.entries[index]\n        if cached.exists():\n            pair = np.load(cached, allow_pickle=False)\n            if pair.shape != (2, self.size, self.size) or pair.dtype != np.uint8:\n                raise ValueError(f"Invalid cached crop: {cached}")\n        else:\n            mask = mask_utils.decode(rle)\n            bounds = crop_bounds(mask, self.context, self.minimum)\n            with Image.open(path) as image:\n                if image.mode != "L" or image.size != (mask.shape[1], mask.shape[0]):\n                    raise ValueError(f"Unexpected grayscale crop image: {path}")\n                pixels = np.asarray(image.crop(bounds).resize(\n                    (self.size, self.size), Image.Resampling.BILINEAR), dtype=np.uint8)\n            x0, y0, x1, y1 = bounds\n            target = np.asarray(Image.fromarray(mask[y0:y1, x0:x1]).resize(\n                (self.size, self.size), Image.Resampling.NEAREST), dtype=np.uint8)\n            pair = np.stack([pixels, target])\n            temporary = cached.with_name(f"{cached.stem}.{os.getpid()}.tmp.npy")\n            try:\n                np.save(temporary, pair, allow_pickle=False)\n                temporary.replace(cached)\n            finally:\n                if temporary.exists():\n                    temporary.unlink()\n        if not np.isin(pair[1], [0, 1]).all():\n            raise ValueError(f"Nonbinary cached crop target: {cached}")\n        x = np.repeat(((pair[0].astype(np.float32) / 255 - 0.5) / 0.5)[None], 3, axis=0)\n        y = pair[1].astype(np.float32)[None]\n        if self.augment:\n            turns = int(torch.randint(4, ()).item())\n            x, y = np.rot90(x, turns, axes=(1, 2)), np.rot90(y, turns, axes=(1, 2))\n            if torch.rand(()).item() < 0.5:\n                x, y = np.flip(x, axis=2), np.flip(y, axis=2)\n        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))\n\n\ndef loader_options(workers, cuda):\n    if type(workers) is not int or workers < 0:\n        raise ValueError("DataLoader worker count must be a nonnegative integer.")\n    options = dict(num_workers=workers, pin_memory=cuda)\n    if workers:\n        options.update(persistent_workers=True, prefetch_factor=2,\n                       multiprocessing_context="spawn")\n    return options\n\n\ndef parallel_refiner(model, device_ids):\n    if not device_ids:\n        return model\n    if len(device_ids) != len(set(device_ids)):\n        raise ValueError("Refiner device IDs must be unique.")\n    if len(device_ids) > 1:\n        return torch.nn.DataParallel(model, device_ids=device_ids, output_device=device_ids[0])\n    return model\n\n\ndef base_refiner(model):\n    return model.module if isinstance(model, torch.nn.DataParallel) else model\n\n\n@torch.inference_mode()\ndef refine_proposals(path, masks, refiner, device, shape, batch_size=16,\n                     crop_size=256, context=1.8, minimum=96):\n    if type(batch_size) is not int or batch_size < 1:\n        raise ValueError("Crop inference batch size must be a positive integer.")\n    if masks.ndim != 3 or masks.shape[1:] != tuple(shape) or not np.isfinite(masks).all():\n        raise ValueError("YOLO masks are not finite full-resolution masks.")\n    proposals = []\n    with Image.open(path) as image:\n        if image.mode != "L" or image.size != (shape[1], shape[0]):\n            raise ValueError(f"Unexpected grayscale inference image: {path}")\n        for start in range(0, len(masks), batch_size):\n            batch, metadata = [], []\n            for mask in masks[start:start + batch_size]:\n                coarse = (mask >= .5).astype(np.uint8)\n                if not coarse.any():\n                    print("WARNING: YOLO returned an empty mask; omitted:", path.name)\n                    continue\n                bounds = crop_bounds(coarse, context, minimum)\n                batch.append(normalized_crop(image, bounds, crop_size))\n                metadata.append((mask_utils.encode(np.asfortranarray(coarse)), bounds))\n            if not batch:\n                continue\n            tensor = torch.from_numpy(np.stack(batch)).to(device)\n            probabilities = refiner(tensor).sigmoid()[:, 0].cpu().numpy()\n            if probabilities.shape != (len(batch), crop_size, crop_size):\n                raise ValueError("Refiner output does not match crop batch shape.")\n            if not np.isfinite(probabilities).all():\n                raise FloatingPointError("Nonfinite refinement probability.")\n            proposals.extend(dict(coarse=rle, bounds=bounds, probability=probability)\n                             for (rle, bounds), probability in zip(metadata, probabilities))\n    return proposals\n')
if str(runtime_dir) not in sys.path:
    sys.path.insert(0, str(runtime_dir))
importlib.invalidate_caches()
from v6_crop_runtime import (
    CachedInstanceCrops, base_refiner, crop_bounds as runtime_crop_bounds,
    loader_options, normalized_crop, parallel_refiner, refine_proposals,
)

LOADER_WORKERS = min(2, max(1, (os.cpu_count() or 2) // 2)) if DEVICE.type == "cuda" else 0
REFINER_GPU_IDS = list(range(min(2, torch.cuda.device_count()))) if DEVICE.type == "cuda" else []
CROP_CACHE_DIR = CV_ROOT / "crop_cache"
print("YOLO device: GPU 0; refiner GPUs:", REFINER_GPU_IDS,
      "CPU loader workers:", LOADER_WORKERS)
if DEVICE.type == "cuda" and len(REFINER_GPU_IDS) < 2:
    print("WARNING: only one CUDA GPU available; refiner will use one GPU.")


In [ ]:
import gc
import yaml
from ultralytics import YOLO

V4 = dict(
    yolo_weights="yolo11s-seg.pt", image_size=1024, batch_size=4,
    yolo_epochs=25, yolo_patience=15,
    yolo_optimizer="AdamW", yolo_lr=0.002, yolo_momentum=0.9,
    yolo_weight_decay=0.0005, yolo_warmup_bias_lr=0.0,
    crop_size=256, crop_context=1.8, minimum_crop=96,
    refiner_encoder="resnet34", refiner_inference_batch=16,
    refiner_epochs=20, refiner_batch=16, refiner_lr=3e-4,
    refiner_patience=8, refiner_min_delta=1e-4,
    tuning_images=8, confidences=[0.15, 0.25],
    refiner_thresholds=[0.5, 0.65], min_areas=[20, 64],
)
RUN_YOLO_TRAINING = True
RUN_REFINER_TRAINING = True
RUN_V4_CALIBRATION = True
RUN_V4_EXPORT = True
REFINER_PRETRAINED = True
ALLOW_CPU_SMOKE = False
YOLO_DEVICE = 0


if V4["yolo_epochs"] < 1 or V4["refiner_epochs"] < 1:
    raise ValueError("Use positive epoch ceilings.")


def v4_seed():
    random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CFG["seed"])


def v4_prepare_dataset():
    directory = OUTPUT_DIR / "yolo_dataset"
    split_manifest = dict(
        annotation_sha256=annotation_sha256, splits=split_names, settings=V4,
        smoke_test=SMOKE_TEST,
        annotation_policy="separate annotator targets; single foreground class",
    )
    saved_path = OUTPUT_DIR / "v4_plan.json"
    if saved_path.exists() and json.loads(saved_path.read_text()) != split_manifest:
        raise ValueError("V4 output has a different plan. Use a new RUN_NAME.")
    write_json(saved_path, split_manifest)
    for phase, label in [("train", "train"), ("tune", "val")]:
        image_dir, label_dir = directory / "images" / label, directory / "labels" / label
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        for index, record in enumerate(split_records[phase]):
            # Distinct label files for each annotator, but all copies remain in one day partition.
            stem = f'{Path(record["file_name"]).stem}__annotation{index}'
            destination = image_dir / f"{stem}.jpeg"
            source = train_lookup[record["file_name"]]
            if not destination.exists():
                destination.symlink_to(source.resolve())
            lines = []
            for annotation in annotations_by_id[record["id"]]:
                polygons = annotation["segmentation"]
                if not isinstance(polygons, list) or len(polygons) != 1:
                    raise ValueError("V4 YOLO conversion requires one polygon per instance; no silent geometry approximation.")
                points = np.asarray(polygons[0], dtype=np.float64)
                if points.size < 6 or points.size % 2 or not np.isfinite(points).all():
                    raise ValueError(f'Invalid polygon {annotation["id"]}')
                points = points.reshape(-1, 2)
                if np.any(points < 0) or np.any(points > 2048):
                    raise ValueError(f'Out-of-bounds polygon {annotation["id"]}')
                normalized = points / 2048
                lines.append("0 " + " ".join(f"{value:.8f}" for value in normalized.ravel()))
            (label_dir / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    path = directory / "data.yaml"
    path.write_text(yaml.safe_dump(dict(
        path=str(directory.resolve()), train="images/train", val="images/val",
        names={0: "filament"},
    )))
    return path


def v4_train_yolo():
    if DEVICE.type != "cuda" and not ALLOW_CPU_SMOKE:
        raise RuntimeError("Enable a GPU before V4 full training.")
    data = v4_prepare_dataset()
    checkpoint = OUTPUT_DIR / "yolo_best.pt"
    if RUN_YOLO_TRAINING:
        if checkpoint.exists():
            raise FileExistsError("Saved YOLO checkpoint exists. Disable training to reuse or choose a new RUN_NAME.")
        v4_seed()
        detector = YOLO(V4["yolo_weights"])
        detector.train(
            data=str(data), epochs=V4["yolo_epochs"], imgsz=V4["image_size"],
            batch=V4["batch_size"], device=YOLO_DEVICE, workers=LOADER_WORKERS,
            patience=V4["yolo_patience"], time=None,
            optimizer=V4["yolo_optimizer"], lr0=V4["yolo_lr"],
            momentum=V4["yolo_momentum"], weight_decay=V4["yolo_weight_decay"],
            warmup_bias_lr=V4["yolo_warmup_bias_lr"],
            seed=CFG["seed"], single_cls=True, mask_ratio=2, overlap_mask=False,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
            fliplr=0.5, flipud=0.0, mosaic=0.3, degrees=10.0,
            scale=0.3, close_mosaic=5, cache=False,
            project=str(OUTPUT_DIR / "detector_runs"), name="yolo11s_seg",
            exist_ok=False, plots=False,
        )
        write_json(OUTPUT_DIR / "yolo_runtime.json", dict(
            last_epoch=detector.trainer.epoch+1, epoch_ceiling=V4["yolo_epochs"],
            stopping_policy="epochs_or_early_stopping", early_stopping=True))
        best = Path(detector.trainer.best)
        if not best.is_file():
            raise FileNotFoundError("YOLO did not produce best.pt.")
        shutil.copyfile(best, checkpoint)
        write_json(OUTPUT_DIR / "yolo_provenance.json", dict(
            sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
            settings=V4, annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
        ))
        del detector
        gc.collect()
    provenance = json.loads((OUTPUT_DIR / "yolo_provenance.json").read_text())
    if (provenance["sha256"] != hashlib.sha256(checkpoint.read_bytes()).hexdigest()
            or provenance["settings"] != V4 or provenance["smoke_test"] != SMOKE_TEST
            or provenance["annotation_sha256"] != annotation_sha256):
        raise ValueError("YOLO checkpoint provenance mismatch.")
    return YOLO(str(checkpoint))


def crop_bounds(mask):
    return runtime_crop_bounds(mask, V4["crop_context"], V4["minimum_crop"])


def crop_input(image, bounds):
    return normalized_crop(image, bounds, V4["crop_size"])


def InstanceCropDataset(records, augment=False):
    entries = [(train_lookup[record["file_name"]], ann_rle(annotation))
               for record in records for annotation in annotations_by_id[record["id"]]]
    return CachedInstanceCrops(entries, CROP_CACHE_DIR, size=V4["crop_size"],
                               context=V4["crop_context"], minimum=V4["minimum_crop"],
                               augment=augment)


def v4_refiner(pretrained):
    return smp.Unet(
        encoder_name=V4["refiner_encoder"], encoder_weights="imagenet" if pretrained else None,
        in_channels=3, classes=1, activation=None,
    ).to(DEVICE)


def v4_train_refiner():
    path = OUTPUT_DIR / "refiner_best.pt"
    if RUN_REFINER_TRAINING:
        if path.exists():
            raise FileExistsError("Saved refiner exists. Disable training or choose a new RUN_NAME.")
        v4_seed()
        refiner = parallel_refiner(v4_refiner(REFINER_PRETRAINED), REFINER_GPU_IDS)
        datasets = {
            phase: InstanceCropDataset(split_records[phase], augment=phase == "train")
            for phase in ["train", "tune"]
        }
        loaders = {
            phase: DataLoader(
                dataset, batch_size=V4["refiner_batch"], shuffle=phase == "train",
                drop_last=phase == "train" and len(dataset) % V4["refiner_batch"] == 1,
                **loader_options(LOADER_WORKERS, DEVICE.type == "cuda"),
            ) for phase, dataset in datasets.items()
        }
        if any(len(loader) == 0 for loader in loaders.values()):
            raise ValueError("Empty crop dataset.")
        optimizer = torch.optim.AdamW(refiner.parameters(), lr=V4["refiner_lr"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=V4["refiner_epochs"]
        )
        amp = DEVICE.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        best, significant_best, stale, history = float("inf"), float("inf"), 0, []
        started = time.monotonic()
        status = "max_epochs"
        for epoch in range(V4["refiner_epochs"]):
            totals = {}
            for phase, loader in loaders.items():
                refiner.train(phase == "train")
                total, count = 0.0, 0
                with torch.set_grad_enabled(phase == "train"):
                    for x, y in tqdm(loader, desc=f"Crop {epoch+1} {phase}"):
                        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                        optimizer.zero_grad(set_to_none=True)
                        with torch.autocast(device_type=DEVICE.type, enabled=amp):
                            logits = refiner(x)
                        loss = segmentation_loss(logits.float(), y.float())
                        if not torch.isfinite(loss):
                            raise FloatingPointError("Nonfinite crop loss.")
                        if phase == "train":
                            scaler.scale(loss).backward()
                            scaler.step(optimizer); scaler.update()
                        total += loss.item()*len(x); count += len(x)
                totals[phase] = total / count
            history.append(dict(epoch=epoch+1, learning_rate=optimizer.param_groups[0]["lr"], **totals))
            print("REFINER:", history[-1], flush=True)
            if totals["tune"] < best:
                best = totals["tune"]
                temporary = path.with_suffix(".tmp.pt")
                torch.save(dict(
                    state_dict={k: v.detach().cpu() for k, v in base_refiner(refiner).state_dict().items()},
                    epoch=epoch+1, settings=V4, annotation_sha256=annotation_sha256,
                    smoke_test=SMOKE_TEST,
                ), temporary)
                temporary.replace(path)
            if totals["tune"] < significant_best - V4["refiner_min_delta"]:
                significant_best, stale = totals["tune"], 0
            else:
                stale += 1
            write_json(OUTPUT_DIR / "refiner_history.json", history)
            scheduler.step()
            if stale >= V4["refiner_patience"]:
                status = "early_stopped"
                break
        write_json(OUTPUT_DIR / "refiner_runtime.json", dict(status=status, completed_epochs=len(history),
            seconds=time.monotonic()-started, stopping_policy="epochs_or_early_stopping"))
        del optimizer, scheduler, scaler, refiner, loaders
        gc.collect()
    if not path.is_file():
        raise FileNotFoundError("Refiner has no completed best checkpoint; inspect training logs.")
    saved = torch.load(path, weights_only=True, map_location="cpu")
    if (saved["settings"] != V4 or saved["annotation_sha256"] != annotation_sha256
            or saved["smoke_test"] != SMOKE_TEST):
        raise ValueError("Refiner checkpoint provenance mismatch.")
    refiner = v4_refiner(False)
    refiner.load_state_dict(saved["state_dict"], strict=True)
    return parallel_refiner(refiner.eval(), REFINER_GPU_IDS)


@torch.inference_mode()
def v4_proposals(path, detector, refiner, confidence):
    result = detector.predict(
        source=str(path), imgsz=V4["image_size"], conf=confidence,
        device=YOLO_DEVICE, agnostic_nms=True, retina_masks=True, verbose=False,
    )[0]
    if result.masks is None:
        return []
    return refine_proposals(
        path, result.masks.data.cpu().numpy(), refiner, DEVICE, FULL_SHAPE,
        batch_size=V4["refiner_inference_batch"], crop_size=V4["crop_size"],
        context=V4["crop_context"], minimum=V4["minimum_crop"],
    )


def v4_instances(proposals, parameters):
    instances, fallback = [], 0
    for proposal in proposals:
        coarse = proposal["coarse"]
        if parameters["mode"] == "yolo":
            rle = coarse
        else:
            x0, y0, x1, y1 = proposal["bounds"]
            probability = np.asarray(Image.fromarray(proposal["probability"]).resize(
                (x1-x0, y1-y0), Image.Resampling.BILINEAR
            ))
            crop = probability >= parameters["refiner_threshold"]
            if not crop.any():
                # Retain an actual detector prediction; never synthesize an empty-image mask.
                fallback += 1
                rle = coarse
            else:
                full = np.zeros(FULL_SHAPE, dtype=np.uint8)
                full[y0:y1, x0:x1] = crop
                rle = mask_utils.encode(np.asfortranarray(full))
        if mask_utils.area(rle) >= parameters["min_area"]:
            instances.append(rle)
    instances, _ = disjoint_instances(instances, FULL_SHAPE, parameters["min_area"])
    return instances, fallback


def v4_scores(names, cache, parameters):
    rows = []
    for name in names:
        predicted, fallback = v4_instances(cache[name], parameters)
        scores = [local_scores(predicted, gt_rles(r["id"])) for r in records_by_name[name]]
        rows.append(dict(filename=name, refiner_fallbacks=fallback, **{
            key: float(np.mean([score[key] for score in scores]))
            for key in scores[0]
        }))
    return {key: float(np.mean([r[key] for r in rows]))
            for key in rows[0] if key != "filename"}, rows


def v4_calibrate(detector, refiner):
    hashes = {name: hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest()
              for name in ["yolo_best.pt", "refiner_best.pt"]}
    path = OUTPUT_DIR / "v4_calibration.json"
    if RUN_V4_CALIBRATION:
        names = list(split_names["tune"])
        random.Random(CFG["seed"]).shuffle(names)
        names = names[:V4["tuning_images"]]
        if not names:
            raise ValueError("No tuning images.")
        trials = []
        for confidence in V4["confidences"]:
            cache = {name: v4_proposals(train_lookup[name], detector, refiner, confidence)
                     for name in tqdm(names, desc=f"Tune confidence={confidence}")}
            for mode in ["yolo", "refined"]:
                thresholds = [None] if mode == "yolo" else V4["refiner_thresholds"]
                for threshold, area in itertools.product(thresholds, V4["min_areas"]):
                    parameters = dict(mode=mode, confidence=confidence,
                                      refiner_threshold=threshold, min_area=area)
                    scores, _ = v4_scores(names, cache, parameters)
                    trials.append(dict(parameters=parameters, scores=scores))
            del cache
        best_by_mode = {
            mode: max((row for row in trials if row["parameters"]["mode"] == mode),
                      key=lambda row: row["scores"]["pq"])
            for mode in ["yolo", "refined"]
        }
        selected = max(best_by_mode.values(), key=lambda row: row["scores"]["pq"])
        saved = dict(
            best_by_mode=best_by_mode, selected=selected, candidates=trials,
            filenames=names, hashes=hashes, settings=V4,
            annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST,
            postprocessing=MASK_POLICY,
        )
        write_json(path, saved)
    else:
        saved = json.loads(path.read_text())
        if (saved.get("postprocessing") != MASK_POLICY
                or saved["hashes"] != hashes or saved["settings"] != V4
                or saved["smoke_test"] != SMOKE_TEST
                or saved["annotation_sha256"] != annotation_sha256):
            raise ValueError("V4 calibration does not match current weights/settings.")
    print("TUNING ONLY, not independent performance:", json.dumps(saved["best_by_mode"], indent=2))
    print("Selected stage:", saved["selected"]["parameters"])
    return saved


def v4_export(detector, refiner, calibration):
    if SMOKE_TEST:
        print("Smoke run: competition export disabled.")
        return None
    if calibration.get("postprocessing") != MASK_POLICY:
        raise ValueError("Re-run calibration with the current disjoint-mask policy; no retraining needed.")
    hashes = {name: hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest()
              for name in ["yolo_best.pt", "refiner_best.pt"]}
    if (calibration["hashes"] != hashes or calibration["settings"] != V4
            or calibration["annotation_sha256"] != annotation_sha256
            or calibration["smoke_test"] != SMOKE_TEST):
        raise ValueError("Export calibration does not match current checkpoints/settings.")
    parameters = calibration["selected"]["parameters"]
    rows, audit = [], []
    for path in tqdm(test_paths, desc="V4 test instances"):
        proposals = v4_proposals(path, detector, refiner, parameters["confidence"])
        instances, fallback = v4_instances(proposals, parameters)
        validate_disjoint(instances, FULL_SHAPE)
        audit.append(dict(filename=path.name, instances=len(instances), refiner_fallbacks=fallback))
        for index, rle in enumerate(instances, 1):
            mask = mask_utils.decode(rle)
            counts = rle["counts"].decode("ascii")
            if mask.shape != FULL_SHAPE or not mask.any():
                raise ValueError("Invalid V4 full-resolution instance.")
            if mask_utils.encode(np.asfortranarray(mask))["counts"].decode("ascii") != counts:
                raise ValueError("V4 RLE roundtrip failed.")
            rows.append(dict(filament_id=f"{path.stem}_{index}", segmentation_rle=counts))
    zeros = [r["filename"] for r in audit if not r["instances"]]
    write_json(OUTPUT_DIR / "test_audit.json", dict(images=audit, zero_detection_images=zeros))
    if not rows:
        print("NOT READY: no instances predicted; no CSV written.")
        return None
    if len(set(r["filament_id"] for r in rows)) != len(rows):
        raise ValueError("Duplicate V4 IDs.")
    validation = validate_submission_rows(rows, {p.stem for p in test_paths}, FULL_SHAPE)
    path = OUTPUT_DIR / "submission.csv"
    temporary = path.with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["filament_id", "segmentation_rle"])
        writer.writeheader(); writer.writerows(rows)
    with temporary.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["filament_id", "segmentation_rle"] or list(reader) != rows:
            raise ValueError("V4 CSV validation failed.")
    temporary.replace(path)
    write_json(OUTPUT_DIR / "submission_manifest.json", dict(
        rows=len(rows), images=len(audit), parameters=parameters,
        hashes=calibration["hashes"], annotation_sha256=annotation_sha256,
        csv_sha256=hashlib.sha256(path.read_bytes()).hexdigest(), smoke_test=False,
        zero_detection_images=zeros, postprocessing=MASK_POLICY, validation=validation,
    ))
    if Path("/kaggle/working").is_dir():
        shutil.copyfile(path, "/kaggle/working/submission.csv")
    if zeros:
        print("WARNING: zero-detection images omitted, no invented masks:", zeros)
    print("CSV PREPARED:", path, "No API upload performed.")
    return path


In [ ]:
"""Day-grouped outer CV with inner tuning for the standalone YOLO/refiner notebook."""

import hashlib
import csv
import json
import math
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils
from scipy.optimize import linear_sum_assignment


def cv_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def make_day_folds(names, seed=2026, folds=5, inner_fraction=0.1):
    names = sorted(names)
    if len(set(names)) != len(names) or folds != 5 or not 0 < inner_fraction < 1:
        raise ValueError("Use unique physical filenames, five folds, and 0 < inner_fraction < 1.")
    by_day = defaultdict(list)
    for name in names:
        datetime.strptime(name[:8], "%Y%m%d")
        by_day[name[:8]].append(name)
    if len(by_day) < 10:
        raise ValueError("Five-fold CV needs at least ten observation days.")
    days = sorted(by_day)
    random.Random(seed).shuffle(days)
    # Largest physical-image groups first; seeded order breaks equal-size ties.
    days.sort(key=lambda day: -len(by_day[day]))
    outer_days, sizes = [[] for _ in range(folds)], [0] * folds
    for day in days:
        fold = min(range(folds), key=lambda index: (sizes[index], index))
        outer_days[fold].append(day)
        sizes[fold] += len(by_day[day])
    plans = []
    for fold in range(folds):
        remaining = sorted(set(days) - set(outer_days[fold]))
        random.Random(seed + fold + 1).shuffle(remaining)
        n_inner = max(1, round(len(remaining) * inner_fraction))
        if n_inner >= len(remaining):
            raise ValueError("Inner tuning leaves no training days.")
        groups = dict(
            train=set(remaining[n_inner:]), tune=set(remaining[:n_inner]),
            holdout=set(outer_days[fold]),
        )
        split = {key: sorted(name for day in selected for name in by_day[day])
                 for key, selected in groups.items()}
        if any(not selected for selected in split.values()):
            raise ValueError("Every CV partition must be nonempty.")
        if set.union(*groups.values()) != set(by_day):
            raise ValueError("CV partitions do not cover the labeled dataset.")
        if any(groups[a] & groups[b] for a, b in
               [("train", "tune"), ("train", "holdout"), ("tune", "holdout")]):
            raise ValueError("Observation-day leakage.")
        plans.append(dict(fold=fold, splits=split))
    outer = [name for plan in plans for name in plan["splits"]["holdout"]]
    if len(outer) != len(names) or set(outer) != set(names):
        raise ValueError("Each physical image must appear in exactly one outer fold.")
    return plans


def save_cv_plan(path, plan):
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != plan:
            raise ValueError("Existing CV plan differs. Choose a new CV_RUN_NAME.")
    else:
        path.write_text(json.dumps(plan, indent=2, allow_nan=False))


def fold_records(plan, records_by_name):
    return {phase: [record for name in names for record in records_by_name[name]]
            for phase, names in plan["splits"].items()}


def validate_fold_result(result, plan, plan_hash):
    if (result["plan_sha256"] != plan_hash or result["fold"] != plan["fold"]
            or result["status"] != "complete"):
        raise ValueError("Fold result provenance/status mismatch.")
    rows = result["outer_rows"]
    expected = plan["splits"]["holdout"]
    actual = [row["filename"] for row in rows]
    if len(actual) != len(set(actual)) or set(actual) != set(expected):
        raise ValueError("Outer rows must cover exactly this fold's physical images.")
    if not result["checkpoint_hashes"] or not result["selected_parameters"]:
        raise ValueError("Missing selected model provenance.")
    for row in rows:
        if row["annotator_sets"] < 1:
            raise ValueError("Missing outer annotator evaluation.")
        for key in ["pq", "dice", "iou"]:
            if not isinstance(row[key], (int, float)) or not math.isfinite(row[key]) or not 0 <= row[key] <= 1:
                raise ValueError("Invalid outer metric.")
    mean = float(np.mean([row["pq"] for row in rows]))
    if not math.isclose(mean, result["outer_mean"]["pq"], abs_tol=1e-12):
        raise ValueError("Outer PQ summary disagrees with per-image rows.")


def summarize_folds(results, plans, plan_hash):
    by_fold = {}
    for result in results:
        fold = result["fold"]
        if fold not in range(5) or fold in by_fold:
            raise ValueError("Unknown or duplicate completed fold.")
        validate_fold_result(result, plans[fold], plan_hash)
        by_fold[fold] = result
    complete = len(by_fold) == 5
    summary = dict(
        plan_sha256=plan_hash, complete=complete, completed_folds=sorted(by_fold),
        missing_folds=sorted(set(range(5)) - set(by_fold)),
        status="complete" if complete else "incomplete: no five-fold aggregate reported",
    )
    if complete:
        fold_pq = [by_fold[fold]["outer_mean"]["pq"] for fold in range(5)]
        rows = [row for fold in range(5) for row in by_fold[fold]["outer_rows"]]
        summary.update(
            fold_mean_pq=float(np.mean(fold_pq)),
            fold_sample_sd_pq=float(np.std(fold_pq, ddof=1)),
            pooled_oof_mean_pq=float(np.mean([row["pq"] for row in rows])),
            physical_images=len(rows),
            note="Per-image metrics average annotators first. Fold SD is descriptive, not a confidence interval. "
                 "These are reused labeled data, not a new untouched final test or leaderboard score.",
        )
    return summary


def collect_fold_results(roots, plans, plan_hash):
    results = {}
    for root in roots:
        for path in sorted(Path(root).rglob("fold_result.json")):
            result = json.loads(path.read_text())
            fold = result["fold"]
            if fold not in range(5):
                raise ValueError(f"Unknown fold in {path}")
            validate_fold_result(result, plans[fold], plan_hash)
            if fold in results and results[fold] != result:
                raise ValueError(f"Conflicting results for fold {fold}; do not select by outer score.")
            results[fold] = result
    return [results[fold] for fold in sorted(results)]


def evaluate_outer(names, parameters, namespace, detector, refiner):
    rows = []
    for name in namespace["tqdm"](names, desc="Outer evaluation: frozen parameters"):
        proposals = namespace["v4_proposals"](
            namespace["train_lookup"][name], detector, refiner, parameters["confidence"],
        )
        predicted, fallbacks = namespace["v4_instances"](proposals, parameters)
        namespace["validate_disjoint"](predicted, namespace["FULL_SHAPE"])
        scores = [namespace["local_scores"](predicted, namespace["gt_rles"](record["id"]))
                  for record in namespace["records_by_name"][name]]
        if not scores:
            raise ValueError("No annotator set for outer image.")
        rows.append(dict(
            filename=name, annotator_sets=len(scores), instances=len(predicted),
            refiner_fallbacks=fallbacks,
            **{key: float(np.mean([score[key] for score in scores])) for key in scores[0]},
        ))
    if not rows:
        raise ValueError("No outer images.")
    means = {key: float(np.mean([row[key] for row in rows]))
             for key in rows[0] if key not in ["filename", "annotator_sets"]}
    return means, rows


def run_cv_fold(namespace, plan, shared_plan, root):
    """Run one isolated fold; outer labels never enter training or calibration."""
    plan_hash = cv_digest(shared_plan)
    directory = Path(root) / f"fold_{plan['fold']}"
    directory.mkdir(parents=True, exist_ok=True)
    result_path = directory / "fold_result.json"
    if result_path.exists():
        result = json.loads(result_path.read_text())
        validate_fold_result(result, plan, plan_hash)
        print("Reusing completed outer evaluation:", result_path)
        return result
    namespace["OUTPUT_DIR"] = directory
    namespace["split_names"] = plan["splits"]
    namespace["split_records"] = fold_records(plan, namespace["records_by_name"])
    namespace["manifest"] = dict(
        seed=namespace["CFG"]["seed"], grouping="observation-day",
        annotation_sha256=namespace["annotation_sha256"], splits=plan["splits"],
    )
    namespace["write_json"](directory / "split_manifest.json", namespace["manifest"])
    for flag, files in [
        ("RUN_YOLO_TRAINING", ["yolo_best.pt", "yolo_provenance.json"]),
        ("RUN_REFINER_TRAINING", ["refiner_best.pt", "refiner_runtime.json"]),
    ]:
        existing = [(directory / name).exists() for name in files]
        if any(existing) and not all(existing):
            raise ValueError(f"Incomplete checkpoint evidence in {directory}; recover it explicitly.")
        namespace[flag] = not all(existing)
    namespace["RUN_V4_CALIBRATION"] = not (directory / "v4_calibration.json").exists()
    detector = refiner = None
    try:
        detector = namespace["v4_train_yolo"]()
        refiner = namespace["v4_train_refiner"]()
        calibration = namespace["v4_calibrate"](detector, refiner)
        means, rows = evaluate_outer(
            plan["splits"]["holdout"], calibration["selected"]["parameters"],
            namespace, detector, refiner,
        )
        result = dict(
            fold=plan["fold"], plan_sha256=plan_hash, status="complete",
            smoke_test=shared_plan["smoke_test"],
            checkpoint_hashes=calibration["hashes"],
            selected_parameters=calibration["selected"]["parameters"],
            inner_selected_scores=calibration["selected"]["scores"],
            outer_mean=means, outer_rows=rows,
        )
        validate_fold_result(result, plan, plan_hash)
        namespace["write_json"](directory / "fold_result.json", result)
        return result
    finally:
        del detector, refiner
        namespace["gc"].collect()
        if namespace["torch"].cuda.is_available():
            namespace["torch"].cuda.empty_cache()


def export_cv_submission(namespace, plan, shared_plan, root):
    """Export a predeclared fold; never select a model by outer score."""
    directory = Path(root) / f"fold_{plan['fold']}"
    result = json.loads((directory / "fold_result.json").read_text())
    validate_fold_result(result, plan, cv_digest(shared_plan))
    namespace.update(
        OUTPUT_DIR=directory, split_names=plan["splits"],
        split_records=fold_records(plan, namespace["records_by_name"]),
        RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,
    )
    detector = refiner = None
    try:
        detector = namespace["v4_train_yolo"]()
        refiner = namespace["v4_train_refiner"]()
        calibration = namespace["v4_calibrate"](detector, refiner)
        if calibration["hashes"] != result["checkpoint_hashes"]:
            raise ValueError("Submission checkpoints differ from the completed fold.")
        return namespace["v4_export"](detector, refiner, calibration)
    finally:
        del detector, refiner
        namespace["gc"].collect()
        if namespace["torch"].cuda.is_available():
            namespace["torch"].cuda.empty_cache()


ENSEMBLE_POLICY = dict(
    method="equal-fold-instance-majority-v1", match_iou=0.5,
    required_fold_votes=3, pixel_votes=3, min_area=20,
)


def fuse_fold_instances(fold_instances, shape):
    """Match at most one instance per fold per cluster, then vote across five folds."""
    if len(fold_instances) != 5:
        raise ValueError("Ensembling requires predictions from exactly five folds.")
    clusters = []
    for instances in fold_instances:
        for rle in instances:
            if tuple(rle["size"]) != tuple(shape) or not mask_utils.area(rle):
                raise ValueError("Invalid ensemble instance shape or empty mask.")
        if not instances:
            continue
        representatives = []
        for cluster in clusters:
            votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),
                        np.zeros(shape, dtype=np.uint8))
            representatives.append(mask_utils.encode(np.asfortranarray(
                (votes * 2 >= len(cluster)).astype(np.uint8))))
        matched = set()
        if representatives:
            overlaps = mask_utils.iou(representatives, instances, [0] * len(instances))
            eligible = overlaps >= ENSEMBLE_POLICY["match_iou"]
            # Maximize valid match count first, then IoU; invalid pairs cannot steal matches.
            weights = eligible * (min(overlaps.shape) + 1 + overlaps)
            rows, columns = linear_sum_assignment(-weights)
            for row, column in zip(rows, columns):
                if eligible[row, column]:
                    clusters[row].append(instances[column])
                    matched.add(column)
        clusters.extend([rle] for index, rle in enumerate(instances) if index not in matched)
    fused = []
    for cluster in clusters:
        if len(cluster) < ENSEMBLE_POLICY["required_fold_votes"]:
            continue
        votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),
                    np.zeros(shape, dtype=np.uint8))
        mask = (votes >= ENSEMBLE_POLICY["pixel_votes"]).astype(np.uint8)
        if int(mask.sum()) >= ENSEMBLE_POLICY["min_area"]:
            fused.append((len(cluster), int(mask.sum()),
                          mask_utils.encode(np.asfortranarray(mask))))
    fused.sort(key=lambda item: (-item[0], -item[1]))
    return [rle for _, _, rle in fused]


def export_cv_ensemble(namespace, plans, shared_plan, root):
    """Reload all five models sequentially and export one validated voted submission."""
    if shared_plan["smoke_test"] or namespace["SMOKE_TEST"]:
        raise ValueError("Smoke runs cannot produce competition exports.")
    if sorted(plan["fold"] for plan in plans) != list(range(5)):
        raise ValueError("All five unique fold plans are required.")
    plans = sorted(plans, key=lambda plan: plan["fold"])
    root = Path(root)
    results = []
    for plan in plans:
        result = json.loads((root / f"fold_{plan['fold']}" / "fold_result.json").read_text())
        validate_fold_result(result, plan, cv_digest(shared_plan))
        if result.get("smoke_test", False):
            raise ValueError("Cannot ensemble smoke checkpoints.")
        results.append(result)
    test_paths = namespace["test_paths"]
    if not test_paths or len({path.stem for path in test_paths}) != len(test_paths):
        raise ValueError("Test image IDs must be nonempty and unique.")
    prediction_paths = []
    for plan, result in zip(plans, results):
        directory = root / f"fold_{plan['fold']}"
        namespace.update(
            OUTPUT_DIR=directory, split_names=plan["splits"],
            split_records=fold_records(plan, namespace["records_by_name"]),
            RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,
        )
        detector = refiner = None
        try:
            detector = namespace["v4_train_yolo"]()
            refiner = namespace["v4_train_refiner"]()
            calibration = namespace["v4_calibrate"](detector, refiner)
            if (calibration["hashes"] != result["checkpoint_hashes"]
                    or calibration["selected"]["parameters"] != result["selected_parameters"]):
                raise ValueError("Ensemble checkpoints/calibration differ from completed fold.")
            parameters = calibration["selected"]["parameters"]
            predictions = {}
            for path in namespace["tqdm"](test_paths, desc=f"Ensemble fold {plan['fold']} test"):
                proposals = namespace["v4_proposals"](
                    path, detector, refiner, parameters["confidence"])
                instances, _ = namespace["v4_instances"](proposals, parameters)
                namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])
                predictions[path.stem] = [
                    dict(size=list(rle["size"]), counts=rle["counts"].decode("ascii"))
                    for rle in instances
                ]
            prediction_path = directory / "ensemble_predictions.json"
            namespace["write_json"](prediction_path, predictions)
            prediction_paths.append(prediction_path)
        finally:
            del detector, refiner
            namespace["gc"].collect()
            if namespace["torch"].cuda.is_available():
                namespace["torch"].cuda.empty_cache()
    predictions = [json.loads(path.read_text()) for path in prediction_paths]
    rows, image_audits = [], []
    for path in test_paths:
        per_fold = [
            [dict(size=rle["size"], counts=rle["counts"].encode("ascii"))
             for rle in fold[path.stem]] for fold in predictions
        ]
        instances = fuse_fold_instances(per_fold, namespace["FULL_SHAPE"])
        instances, ownership = namespace["disjoint_instances"](
            instances, namespace["FULL_SHAPE"], ENSEMBLE_POLICY["min_area"])
        namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])
        for rle in instances:
            if mask_utils.encode(np.asfortranarray(mask_utils.decode(rle)))["counts"] != rle["counts"]:
                raise ValueError("Ensemble RLE roundtrip failed.")
        image_audits.append(dict(
            filename=path.name, fold_instances=[len(items) for items in per_fold],
            instances=len(instances), ownership=ownership,
        ))
        rows.extend(dict(filament_id=f"{path.stem}_{index}",
                         segmentation_rle=rle["counts"].decode("ascii"))
                    for index, rle in enumerate(instances, 1))
    audit = namespace["validate_submission_rows"](
        rows, {path.stem for path in test_paths}, namespace["FULL_SHAPE"])
    destination = root / "submission.csv"
    temporary = root / "submission.ensemble.tmp.csv"
    fields = ["filament_id", "segmentation_rle"]
    try:
        with temporary.open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
        with temporary.open(newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != fields or list(reader) != rows:
                raise ValueError("Ensemble CSV readback mismatch.")
        namespace["write_json"](root / "ensemble_audit.json", dict(
            policy=ENSEMBLE_POLICY, plan_sha256=cv_digest(shared_plan),
            folds=[dict(fold=r["fold"], hashes=r["checkpoint_hashes"],
                        parameters=r["selected_parameters"]) for r in results],
            validation=audit, images=image_audits,
            submission_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),
        ))
        temporary.replace(destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    kaggle_root = Path("/kaggle/working")
    if kaggle_root.is_dir() and destination.resolve() != (kaggle_root / "submission.csv").resolve():
        namespace["shutil"].copy2(destination, kaggle_root / "submission.csv")
    namespace["OUTPUT_DIR"] = root
    if audit["zero_detection_images"]:
        print("WARNING: zero-detection images omitted, no invented masks:",
              audit["zero_detection_images"])
    return destination


## Five day-grouped partitions
The fold-count table distinguishes unique physical images, annotator sets, and
instance crops. Outer folds never control checkpoints or calibration. Fold0's
submission is predeclared, not selected from outer scores. Changing settings
requires a new CV_RUN_NAME. All five folds are selected by default.


In [ ]:
if (not FOLDS_TO_RUN or len(set(FOLDS_TO_RUN)) != len(FOLDS_TO_RUN)
        or any(type(fold) is not int or fold not in range(5) for fold in FOLDS_TO_RUN)):
    raise ValueError("Choose distinct integer fold indices from 0 through 4.")
cv_plans = make_day_folds(records_by_name, CFG["seed"], inner_fraction=INNER_TUNE_FRACTION)
cv_plan = dict(
    seed=CFG["seed"], annotation_sha256=annotation_sha256, settings=V4,
    inner_tune_fraction=INNER_TUNE_FRACTION, folds=cv_plans,
    mask_policy=MASK_POLICY, smoke_test=SMOKE_TEST,
    ensemble_policy=ENSEMBLE_POLICY, training_stopping_policy="epochs_or_early_stopping",
    crop_runtime_sha256='da3a220e1ea9a83d13d14b282a9652a1da0d5355cd553d6b8b271e1dba192f3b',
    training_code_sha256='a72b2192ba02ab783da50cdfb9f936f9b0c3747f07def8e35d39f8c8b1166c3f',
)
save_cv_plan(CV_ROOT / "cv_plan.json", cv_plan)
cv_plan_hash = cv_digest(cv_plan)
for plan in cv_plans:
    entries = fold_records(plan, records_by_name)
    print("Fold", plan["fold"], {
        phase: dict(
            physical_images=len(plan["splits"][phase]),
            annotator_sets=len(selected),
            instance_crops=sum(len(annotations_by_id[r["id"]]) for r in selected),
        ) for phase, selected in entries.items()
    })
print("Selected folds:", FOLDS_TO_RUN, "Plan SHA256:", cv_plan_hash)
with (CV_ROOT / "requirements-actual.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)


## Five-fold training without wall-clock limits
Each selected fold allows 25 detector and 20 refiner epochs, with early stopping
retained (YOLO validation-fitness patience 15; refiner tuning-loss patience 8).
There is no wall-time allocation. Best-checkpoint selection and cosine refiner
scheduling remain active. Completed matching fold results can be reused.


In [ ]:

for fold in FOLDS_TO_RUN:
    result = run_cv_fold(globals(), cv_plans[fold], cv_plan, CV_ROOT)
    print("OUTER FOLD", fold, result["outer_mean"],
          "SMOKE ONLY" if SMOKE_TEST else "not a leaderboard score")


## Five-fold summary and manual submission
Only complete matching folds enter the collector; full mean/SD requires all five.
No outer score selects ensemble weights. To collect saved runs without training,
execute setup/plan and summary, not the training cell.
The next cell reloads all five models sequentially, predicts every test image,
matches instances and applies equal-fold majority voting. It validates one final
disjoint CSV and records every model's checkpoint hashes in ensemble_audit.json.
The per-fold OOF metrics evaluate individual models, not the test ensemble.


In [ ]:

cv_results = collect_fold_results([CV_ROOT, *map(Path, COLLECT_ROOTS)], cv_plans, cv_plan_hash)
cv_summary = summarize_folds(cv_results, cv_plans, cv_plan_hash)
cv_summary["smoke_test"] = SMOKE_TEST
write_json(CV_ROOT / "cv_summary.json", cv_summary)
print(json.dumps(cv_summary, indent=2))
print("Save full CV evidence:", CV_ROOT)


In [ ]:

if not cv_summary["complete"]:
    raise RuntimeError("All five folds must finish before ensemble submission export.")
submission_path = export_cv_ensemble(globals(), cv_plans, cv_plan, CV_ROOT)
elapsed = time.monotonic() - CV_STARTED
write_json(CV_ROOT / "turnaround.json", dict(
    elapsed_seconds=elapsed, training_stopping_policy="epochs_or_early_stopping",
    submission_folds=list(range(5)),
    ensemble=True, ensemble_policy=ENSEMBLE_POLICY,
    detector_epochs_per_fold=V4["yolo_epochs"],
    refiner_epochs_per_fold=V4["refiner_epochs"],
))
print("Elapsed minutes:", round(elapsed / 60, 1))
print("FINAL CSV PREPARED:", submission_path,
      "Kaggle root copy: /kaggle/working/submission.csv. No upload performed.")
